In [1]:
import os
for f in os.listdir('../data/DOM-data'):
    print(f)

.DS_Store
DOM Equations.docx
Example.xlsx
input data
Output_order_level_data.csv
output_order_sku_level_data.csv
~$M Equations.docx


In [2]:
for f in os.listdir('../data/DOM-data/input data'):
    print(f)

input_capacity_planning.csv
input_dock_capacity.csv
input_order data.csv
input_shipping_cost_data.csv
input_throughput_capacity.csv


In [3]:
import pandas as pd
df_order = pd.read_csv('../data/DOM-data/Output_order_level_data.csv')
df_order.head()

,SalesDocument/GroupingIndicator,IsDivert,LoadNumber,DefaultDC,RecommendedDC,DefaultPGI,ExpectedPGI,RequestedDeliveryDate,CustomerGroup5Description,CustomerNumberShipTo,...,PayloadPerc_Default,PayloadWaste_Default,PayloadWaste_Divert,Additional_PayloadWaste,Additional_ShippingCost,Additional_Fines,Net_Cost/Saving,CostperRevenueRatio,IsSavingPenalty,DCWisePenaltyRanking
0,5485387044,Default,U600105964,5083,5083,6/27/24,NaN,7/1/24,NaN,1086983.0,...,0.000000,3819.000000,3819.000000,0.0,0,0.0,0.0,0.0,N,1
1,5484985692,Default,U600105464,5083,5083,6/24/24,6/24/24,6/25/24,NaN,6980752.0,...,0.608994,826.977186,826.977186,0.0,0,0.0,0.0,0.0,N,1
2,5484810411,Default,U600104570,5083,5083,6/24/24,6/24/24,6/27/24,NaN,1093380.0,...,0.698462,1164.538836,1164.538836,0.0,0,0.0,0.0,0.0,N,1
3,5485261850,Default,U600105897,5083,5083,6/24/24,6/24/24,6/25/24,NaN,6534722.0,...,1.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,N,1
4,5485126048,Default,U600105478,5083,5083,6/24/24,6/24/24,6/26/24,NaN,5509100.0,...,0.514795,1413.402997,1413.402997,0.0,0,0.0,0.0,0.0,N,1


In [4]:
import pandas as pd

df_order = pd.read_csv('../data/DOM-data/input data/input_order data.csv')
print(df_order['Unnamed: 38'].describe())
print(df_order['Unnamed: 38'].head(10))

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Unnamed: 38, dtype: float64
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: Unnamed: 38, dtype: float64


In [5]:
import pandas as pd
df_dock = pd.read_csv('../data/DOM-data/input data/input_dock_capacity.csv')
print(df_dock.columns.tolist())
print("Total rows:", df_dock.shape[0])

df_ship = pd.read_csv('../data/DOM-data/input data/input_shipping_cost_data.csv')
print(df_ship.columns.tolist())
print("Total rows:", df_ship.shape[0])

df_through = pd.read_csv('../data/DOM-data/input data/input_throughput_capacity.csv')
print(df_through.columns.tolist())
print("Total rows:", df_through.shape[0])


print(df_dock[['Plant','Date']].drop_duplicates().shape[0], "vs", df_dock.shape[0])
print(df_ship[['Plant','OrigZip3','TargetZip']].drop_duplicates().shape[0], "vs", df_ship.shape[0])
print(df_through[['Plant','transportationplanningdate']].drop_duplicates().shape[0], "vs", df_through.shape[0])

['Plant', 'Date', 'Modified_Date', 'SnapshotDate', 'Comments', 'ActiveFlag', 'InboundAppointments', 'Dock_Capacity', 'TotalAppointments', 'isChanged', 'Dock_Booked', 'Dock_Remaining', 'Report_Run_Date']
Total rows: 480
['TargetZip', 'OrigZip3', 'CostPerLoadAmbientOutboundMoves', 'FuelSurchargeAmbient', 'Distance', 'Shipping_Cost', 'Plant']
Total rows: 12922
['Plant', 'transportationplanningdate', 'util_case_picks', 'util_pallets', 'order_count', 'Day', 'Report_Run_Date']
Total rows: 530
480 vs 480
12922 vs 12922
530 vs 530


In [6]:
import pandas as pd
df_order = pd.read_csv('../data/DOM-data/input data/input_order data.csv')
df_capacity = pd.read_csv('../data/DOM-data/input data/input_capacity_planning.csv')
df_capacity_renamed = df_capacity.rename(columns={
    'LocationID': 'Plant',
    'MaterialID': 'MaterialNumber',
    'DATE': 'transportationplanningdate'
})
df_order['transportationplanningdate'] = pd.to_datetime(df_order['transportationplanningdate'], format='%m/%d/%y')
df_capacity_renamed['transportationplanningdate'] = pd.to_datetime(df_capacity_renamed['transportationplanningdate'], format='%Y-%m-%d')

df_baseline = df_order.merge(
    df_capacity_renamed[['Plant','MaterialNumber','transportationplanningdate','Available_inventory']],
    on=['Plant','MaterialNumber','transportationplanningdate'],
    how='left'
)
# Cases that can actually be filled at the default DC = min(ordered qty, available inventory)
df_baseline['Filled_Qty_Default'] = df_baseline[['OrderedQty_converted','Available_inventory']].min(axis=1)

df_baseline['Filled_Qty_Default'] = df_baseline['Filled_Qty_Default'].fillna(0)

df_baseline['FillRate_Default'] = df_baseline['Filled_Qty_Default'] / df_baseline['OrderedQty_converted']
df_baseline['FillRate_Default'] = df_baseline['FillRate_Default'].fillna(0)  # in case OrderedQty is 0

total_ordered = df_baseline['OrderedQty_converted'].sum()
total_filled = df_baseline['Filled_Qty_Default'].sum()
overall_fill_rate = total_filled / total_ordered

print(f"Total ordered qty: {total_ordered:,.0f}")
print(f"Total filled qty (default DC only): {total_filled:,.0f}")
print(f"Overall fill rate (default assignment): {overall_fill_rate:.2%}")
###
df_baseline['Unfilled_Qty_Default'] = df_baseline['OrderedQty_converted'] - df_baseline['Filled_Qty_Default']

# Penalty = unfilled qty * price * penalty rate (from equation doc logic)
df_baseline['Penalty_Cost_Default'] = (
    df_baseline['Unfilled_Qty_Default'] * 
    df_baseline['Order_SKU_Revenue'] * 
    df_baseline['Penaltyforpotentialcuts']
)
df_baseline['Penalty_Cost_Default'] = df_baseline['Penalty_Cost_Default'].fillna(0)

total_penalty = df_baseline['Penalty_Cost_Default'].sum()
print(f"Total penalty cost (default assignment): {total_penalty:,.2f}")
###
df_dock = pd.read_csv('../data/DOM-data/input data/input_dock_capacity.csv')
df_ship = pd.read_csv('../data/DOM-data/input data/input_shipping_cost_data.csv')
df_through = pd.read_csv('../data/DOM-data/input data/input_throughput_capacity.csv')

df_baseline_ship = df_baseline.merge(
    df_ship[['Plant','TargetZip','Shipping_Cost']],
    left_on=['Plant','ZipCode'],
    right_on=['Plant','TargetZip'],
    how='left'
)
default_shipping_correct = df_baseline_ship.drop_duplicates(subset=['Group_Flag','Plant'])['Shipping_Cost'].sum()
print("Corrected default shipping cost:", f"{default_shipping_correct:,.2f}")

print("=== DEFAULT-ASSIGNMENT BASELINE ===")
print(f"Total ordered qty: {total_ordered:,.0f}")
print(f"Total filled qty: {total_filled:,.0f}")
print(f"Overall fill rate: {overall_fill_rate:.2%}")
print(f"Total penalty cost: {total_penalty:,.2f}")
print(f"Total shipping cost: {default_shipping_correct:,.2f}")
print(f"Number of reassignments: 0 (by definition)")


Total ordered qty: 2,345,613
Total filled qty (default DC only): 2,247,512
Overall fill rate (default assignment): 95.82%
Total penalty cost (default assignment): 7,600,586.25
Corrected default shipping cost: 1,386,508.00
=== DEFAULT-ASSIGNMENT BASELINE ===
Total ordered qty: 2,345,613
Total filled qty: 2,247,512
Overall fill rate: 95.82%
Total penalty cost: 7,600,586.25
Total shipping cost: 1,386,508.00
Number of reassignments: 0 (by definition)


In [7]:

needs_reassignment = df_baseline[df_baseline['Unfilled_Qty_Default'] > 0].copy()
print("Order-SKU lines needing reassignment:", needs_reassignment.shape[0])
print("Total unfilled qty to try to cover:", needs_reassignment['Unfilled_Qty_Default'].sum())

alt_inventory = df_capacity_renamed[['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
alt_inventory = alt_inventory.rename(columns={'Plant':'Alt_Plant'})
print(alt_inventory.head())
print("Rows:", alt_inventory.shape[0])

Order-SKU lines needing reassignment: 1340
Total unfilled qty to try to cover: 98101.0
   Alt_Plant  MaterialNumber transportationplanningdate  Available_inventory
0       5081        12321261                 2024-06-20                  0.0
1       5081        12321261                 2024-06-21                  0.0
2       5081        12321261                 2024-06-22                  0.0
3       5081        12321261                 2024-06-23                  0.0
4       5081        12321261                 2024-06-24                  0.0
Rows: 377504


In [8]:
# Build a mutable dict: (Alt_Plant, MaterialNumber, date) -> remaining inventory
inventory_pool = alt_inventory.set_index(['Alt_Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()

results = []

for idx, row in needs_reassignment.iterrows():
    sku = row['MaterialNumber']
    date = row['transportationplanningdate']
    default_plant = row['Plant']
    shortfall = row['Unfilled_Qty_Default']
    
    # find candidate alt DCs for this SKU+date, excluding the default plant, with inventory > 0
    candidates = {
        plant: inv for (plant, m, d), inv in inventory_pool.items()
        if m == sku and d == date and plant != default_plant and inv > 0
    }
    
    if candidates:
        # pick the DC with the most available inventory (greedy choice)
        best_plant = max(candidates, key=candidates.get)
        available = candidates[best_plant]
        filled_from_alt = min(shortfall, available)
        
        # deduct from the pool
        inventory_pool[(best_plant, sku, date)] -= filled_from_alt
        
        results.append({
            'Group_Flag': row['Group_Flag'],
            'MaterialNumber': sku,
            'Default_Plant': default_plant,
            'Reassigned_Plant': best_plant,
            'Filled_From_Alt': filled_from_alt,
            'Still_Unfilled': shortfall - filled_from_alt
        })
    else:
        results.append({
            'Group_Flag': row['Group_Flag'],
            'MaterialNumber': sku,
            'Default_Plant': default_plant,
            'Reassigned_Plant': None,
            'Filled_From_Alt': 0,
            'Still_Unfilled': shortfall
        })

df_greedy_results = pd.DataFrame(results)
print(df_greedy_results.shape[0])
df_greedy_results.head(10)

1340


,Group_Flag,MaterialNumber,Default_Plant,Reassigned_Plant,Filled_From_Alt,Still_Unfilled
0,5484913123,12180656,5083,NaN,0.0,7.0
1,5484913123,12154830,5083,NaN,0.0,9.0
2,5484913123,12154829,5083,5773.0,120.0,0.0
3,5484913123,12260380,5083,NaN,0.0,30.0
4,5484930889,11005773,5620,5641.0,32.0,0.0
5,5484930889,12421245,5620,5420.0,38.0,0.0
6,5484495964,12484724,5410,5490.0,720.0,0.0
7,5484845895,12420977,5620,5410.0,6.0,0.0
8,5485088170,12434028,5420,5641.0,1.0,0.0
9,5485088170,12039865,5420,5490.0,480.0,0.0


In [9]:
# Computing Overall greedy baseline metrics
total_filled_from_alt = df_greedy_results['Filled_From_Alt'].sum()
total_still_unfilled = df_greedy_results['Still_Unfilled'].sum()
num_reassignments = (df_greedy_results['Reassigned_Plant'].notna()).sum()

# New total filled = what was already filled at default + what greedy recovered from alt DCs
total_filled_greedy = total_filled + total_filled_from_alt
overall_fill_rate_greedy = total_filled_greedy / total_ordered

print(f"Number of reassignments made: {num_reassignments}")
print(f"Additional qty filled via reassignment: {total_filled_from_alt:,.0f}")
print(f"Remaining unfilled qty: {total_still_unfilled:,.0f}")
print(f"New total filled qty (greedy): {total_filled_greedy:,.0f}")
print(f"New overall fill rate (greedy): {overall_fill_rate_greedy:.2%}")

Number of reassignments made: 849
Additional qty filled via reassignment: 58,110
Remaining unfilled qty: 39,991
New total filled qty (greedy): 2,305,622
New overall fill rate (greedy): 98.30%


In [10]:
# merge greedy results back onto needs_reassignment to get penalty rate + price per line
df_greedy_full = needs_reassignment.merge(
    df_greedy_results[['Group_Flag','MaterialNumber','Reassigned_Plant','Filled_From_Alt','Still_Unfilled']],
    on=['Group_Flag','MaterialNumber'],
    how='left'
)

df_greedy_full['Penalty_Cost_Greedy'] = (
    df_greedy_full['Still_Unfilled'] *
    df_greedy_full['Order_SKU_Revenue'] *
    df_greedy_full['Penaltyforpotentialcuts']
)
df_greedy_full['Penalty_Cost_Greedy'] = df_greedy_full['Penalty_Cost_Greedy'].fillna(0)

# lines that were already fully filled at default DC have zero penalty (not in needs_reassignment)
total_penalty_greedy = df_greedy_full['Penalty_Cost_Greedy'].sum()
print(f"Total penalty cost (greedy): {total_penalty_greedy:,.2f}")

Total penalty cost (greedy): 2,731,905.98


In [11]:
df_baseline['price_per_case'] = (df_baseline['Order_SKU_Revenue'] / df_baseline['OrderedQty_converted']).fillna(0)
default_revenue = (df_baseline['price_per_case'] * df_baseline['Filled_Qty_Default']).sum()

df_greedy_full = df_greedy_full.merge(
    df_baseline[['Group_Flag','MaterialNumber','price_per_case']],
    on=['Group_Flag','MaterialNumber'], how='left'
)
extra_revenue_from_reassignment = (df_greedy_full['price_per_case'] * df_greedy_full['Filled_From_Alt']).sum()
greedy_revenue = default_revenue + extra_revenue_from_reassignment

In [12]:
# --- Shipping (FIXED — one charge per (order, plant) actually used) ---
default_used = df_baseline[df_baseline['Filled_Qty_Default'] > 0][['Group_Flag','Plant']].drop_duplicates()
reassigned_used = df_greedy_full[df_greedy_full['Filled_From_Alt'] > 0][['Group_Flag','Reassigned_Plant']]
reassigned_used = reassigned_used.rename(columns={'Reassigned_Plant':'Plant'}).drop_duplicates()
plants_used = pd.concat([default_used, reassigned_used]).drop_duplicates(subset=['Group_Flag','Plant'])
plants_used = plants_used.merge(df_baseline[['Group_Flag','ZipCode']].drop_duplicates(), on='Group_Flag', how='left')
plants_used = plants_used.merge(
    df_ship[['Plant','TargetZip','Shipping_Cost']],
    left_on=['Plant','ZipCode'], right_on=['Plant','TargetZip'], how='left'
)
plants_used['Shipping_Cost'] = plants_used['Shipping_Cost'].fillna(0)
total_shipping_greedy = plants_used['Shipping_Cost'].sum()

# --- Profit (NEW) ---
greedy_profit = greedy_revenue - total_penalty_greedy - total_shipping_greedy



In [13]:
# --- Summary ---
print("=== GREEDY / SEQUENTIAL-REASSIGNMENT BASELINE ===")
print(f"Total ordered qty: {total_ordered:,.0f}")
print(f"Total filled qty: {total_filled_greedy:,.0f}")
print(f"Overall fill rate: {overall_fill_rate_greedy:.2%}")
print(f"Total revenue: {greedy_revenue:,.2f}")
print(f"Total penalty cost: {total_penalty_greedy:,.2f}")
print(f"Total shipping cost: {total_shipping_greedy:,.2f}")
print(f"Total profit (revenue - penalty - shipping): {greedy_profit:,.2f}")
print(f"Number of reassignments: {num_reassignments}")

=== GREEDY / SEQUENTIAL-REASSIGNMENT BASELINE ===
Total ordered qty: 2,345,613
Total filled qty: 2,305,622
Overall fill rate: 98.30%
Total revenue: 87,092,157.33
Total penalty cost: 2,731,905.98
Total shipping cost: 3,221,692.00
Total profit (revenue - penalty - shipping): 81,138,559.36
Number of reassignments: 849


## MILP Formulation

In [14]:
# Formulation 
import pulp

# Start with a small, manageable slice for correctness testing
sample_orders = df_order['Group_Flag'].unique()[:20]
df_sample = df_baseline_ship[df_baseline_ship['Group_Flag'].isin(sample_orders)].copy()

print("Sample order-SKU lines:", df_sample.shape[0])
print("Unique orders in sample:", df_sample['Group_Flag'].nunique())
print("Unique DCs (default only, for now):", df_sample['Plant'].nunique())

Sample order-SKU lines: 553
Unique orders in sample: 20
Unique DCs (default only, for now): 7


In [15]:
sample_dates = df_sample['transportationplanningdate'].unique()
sample_skus = df_sample['MaterialNumber'].unique()
candidate_dcs = df_capacity_renamed[
    (df_capacity_renamed['MaterialNumber'].isin(sample_skus)) &
    (df_capacity_renamed['transportationplanningdate'].isin(sample_dates))
]
candidate_dcs = candidate_dcs[['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()

print("Candidate DC-SKU-date rows (date-filtered):", candidate_dcs.shape[0])
print("Unique DCs appearing as candidates:", candidate_dcs['Plant'].nunique())

# Get zip for each sample order
order_zips = df_sample[['Group_Flag','ZipCode']].drop_duplicates()

# Cross with all candidate DCs' shipping rates to that zip
ship_lookup = df_ship[df_ship['Plant'].isin(candidate_dcs['Plant'].unique())][['Plant','TargetZip','Shipping_Cost']]

sample_ship_costs = order_zips.merge(ship_lookup, left_on='ZipCode', right_on='TargetZip', how='left')
print("Order x candidate-DC shipping cost rows:", sample_ship_costs.shape[0])
print("Missing shipping cost rows:", sample_ship_costs['Shipping_Cost'].isna().sum())

Candidate DC-SKU-date rows (date-filtered): 13914
Unique DCs appearing as candidates: 12
Order x candidate-DC shipping cost rows: 240
Missing shipping cost rows: 0


In [16]:
# Declaring Decision Variables
import pulp

prob = pulp.LpProblem("DOM_Sample", pulp.LpMaximize)

orders = df_sample['Group_Flag'].unique()
dcs = candidate_dcs['Plant'].unique()

# x[o,d] = 1 if order o assigned to DC d
x = {}
for o in orders:
    for d in dcs:
        x[(o,d)] = pulp.LpVariable(f"x_{o}_{d}", cat='Binary')

# f[o,s,d] = cases of SKU s in order o fulfilled from DC d
f = {}
for _, row in df_sample.iterrows():
    o = row['Group_Flag']
    s = row['MaterialNumber']
    for d in dcs:
        f[(o,s,d)] = pulp.LpVariable(f"f_{o}_{s}_{d}", lowBound=0, cat='Continuous')

print("Number of x variables:", len(x))
print("Number of f variables:", len(f))

Number of x variables: 240
Number of f variables: 6636


In [17]:
# Checking Missing Penalty Rates
print("Rows with missing penalty rate:", df_sample['Penaltyforpotentialcuts'].isna().sum(), "out of", df_sample.shape[0])
print(df_sample.groupby('IsTopCust')['Penaltyforpotentialcuts'].apply(lambda x: x.isna().sum()))

Rows with missing penalty rate: 134 out of 553
IsTopCust
N    134
Y      0
Name: Penaltyforpotentialcuts, dtype: int64


Fill NaNs at the source, before creating the lookup dicts
Confirmed: missing penalty rate corresponds exactly to IsTopCust == 'N' (134/553 in sample).
Non-top customers have no penalty clause defined — treating missing as 0 is a business-accurate
assumption, not a data quality workaround.

In [18]:

df_sample_clean = df_sample.copy()
df_sample_clean['Order_SKU_Revenue'] = df_sample_clean['Order_SKU_Revenue'].fillna(0)
df_sample_clean['Penaltyforpotentialcuts'] = df_sample_clean['Penaltyforpotentialcuts'].fillna(0)
df_sample_clean['OrderedQty_converted'] = df_sample_clean['OrderedQty_converted'].fillna(0)

price = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['Order_SKU_Revenue'].to_dict()
penalty_rate = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['Penaltyforpotentialcuts'].to_dict()
demand = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['OrderedQty_converted'].to_dict()

sample_ship_costs_clean = sample_ship_costs.copy()
sample_ship_costs_clean['Shipping_Cost'] = sample_ship_costs_clean['Shipping_Cost'].fillna(0)
ship_cost = sample_ship_costs_clean.set_index(['Group_Flag','Plant'])['Shipping_Cost'].to_dict()

In [19]:
# Building the Objective Function
# Corrected per-case price (Order_SKU_Revenue was total line revenue, not per-case price)
df_sample_clean['price_per_case'] = df_sample_clean['Order_SKU_Revenue'] / df_sample_clean['OrderedQty_converted']
price = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['price_per_case'].to_dict()

# Revenue term
revenue = pulp.lpSum(
    price.get((o,s), 0) * f[(o,s,d)]
    for (o,s,d) in f
)

# Penalty term: penalty_rate * price * (demand - total filled across all DCs)
unfilled_by_line = {}
for (o,s) in set((k[0],k[1]) for k in f):
    unfilled_by_line[(o,s)] = demand.get((o,s),0) - pulp.lpSum(f[(o,s,d)] for d in dcs if (o,s,d) in f)

penalty = pulp.lpSum(
    penalty_rate.get((o,s),0) * price.get((o,s),0) * unfilled_by_line[(o,s)]
    for (o,s) in unfilled_by_line
)

# Shipping cost term
shipping = pulp.lpSum(
    ship_cost.get((o,d), 0) * x[(o,d)]
    for (o,d) in x
)

prob.setObjective(revenue - penalty - shipping)
print("Objective function updated with corrected price.")

Objective function updated with corrected price.


In [20]:
# Adding Constraints
from collections import defaultdict
# Each order assigned to exactly one DC
# Test 1: only Constraint 1 (one DC per order)
for o in orders:
    prob += pulp.lpSum(x[(o,d)] for d in dcs) == 1, f"OneDC_{o}"

prob.solve()
print("After Constraint 1 only:", pulp.LpStatus[prob.status])
# Fulfillment only allowed at the assigned DC
for (o,s,d) in f:
    prob += f[(o,s,d)] <= demand.get((o,s),0) * x[(o,d)], f"OnlyIfAssigned_{o}_{s}_{d}"

prob.solve()
print("After Constraint 2:", pulp.LpStatus[prob.status])
# Cannot exceed ordered quantity
for (o,s) in unfilled_by_line:
    prob += pulp.lpSum(f[(o,s,d)] for d in dcs if (o,s,d) in f) <= demand.get((o,s),0), f"MaxDemand_{o}_{s}"
prob.solve()
print("After Constraint 3:", pulp.LpStatus[prob.status])

# Inventory limit at each DC-SKU-date
candidate_dcs['Available_inventory'] = candidate_dcs['Available_inventory'].clip(lower=0)
inv_lookup = candidate_dcs.set_index(['Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()
inv_groups = defaultdict(list)
for (o,s,d) in f:
    date = df_sample_clean[(df_sample_clean['Group_Flag']==o) & (df_sample_clean['MaterialNumber']==s)]['transportationplanningdate'].iloc[0]
    inv_groups[(s,d,date)].append((o,s,d))
for (s,d,date), var_list in inv_groups.items():
    available = inv_lookup.get((d,s,date), 0)
    prob += pulp.lpSum(f[k] for k in var_list) <= available, f"Inventory_{s}_{d}_{date}"

prob.solve()
print("After Constraint 4 (fixed):", pulp.LpStatus[prob.status])

# Dock capacity limit at each DC-date
# Test 5: add Constraint 5 (dock, with capacity > 0 fix)
dock_lookup = df_dock.set_index(['Plant','Date'])['Dock_Capacity'].to_dict()
order_dates = df_sample_clean.set_index('Group_Flag')['transportationplanningdate'].to_dict()
dock_groups = defaultdict(list)
for (o,d) in x:
    date = order_dates[o]
    dock_groups[(d,date)].append((o,d))
for (d,date), var_list in dock_groups.items():
    capacity = dock_lookup.get((d,date), None)
    if capacity is not None and capacity > 0:
        prob += pulp.lpSum(x[k] for k in var_list) <= capacity, f"Dock_{d}_{date}"

prob.solve()
print("After Constraint 5:", pulp.LpStatus[prob.status])

After Constraint 1 only: Unbounded
After Constraint 2: Optimal
After Constraint 3: Optimal
After Constraint 4 (fixed): Optimal
After Constraint 5: Optimal


In [21]:
from collections import Counter

orders_per_date = Counter(order_dates[o] for o in orders)

for date, count in orders_per_date.items():
    total_capacity_that_date = sum(dock_lookup.get((d,date), 0) for d in dcs)
    print(f"Date {date.date()}: {count} orders need assignment, total dock capacity across DCs = {total_capacity_that_date}")

Date 2024-06-26: 7 orders need assignment, total dock capacity across DCs = 0
Date 2024-06-25: 4 orders need assignment, total dock capacity across DCs = 0
Date 2024-06-24: 2 orders need assignment, total dock capacity across DCs = 0
Date 2024-06-27: 5 orders need assignment, total dock capacity across DCs = 0
Date 2024-07-02: 1 orders need assignment, total dock capacity across DCs = 0
Date 2024-06-28: 1 orders need assignment, total dock capacity across DCs = 0


In [22]:
prob.solve()
total_filled = sum(pulp.value(f[k]) for k in f if pulp.value(f[k]) > 0)
print("Total cases filled by optimizer:", total_filled)
print("Total demand across sample:", df_sample_clean['OrderedQty_converted'].sum())

# check how many DCs each order got assigned to (should be exactly 1 each)
for o in orders:
    assigned = [d for d in dcs if pulp.value(x[(o,d)]) == 1]
    print(f"Order {o}: assigned to {assigned}")

Total cases filled by optimizer: 53487.0
Total demand across sample: 54744
Order 5484913123: assigned to [np.int64(5773)]
Order 5484930889: assigned to [np.int64(5641)]
Order 5485421377: assigned to [np.int64(5641)]
Order 5485503065: assigned to [np.int64(5490)]
Order 5485506985: assigned to [np.int64(5385)]
Order 5484495964: assigned to [np.int64(5490)]
Order 5484845895: assigned to [np.int64(5620)]
Order 5485088170: assigned to [np.int64(5410)]
Order 5485344857: assigned to [np.int64(5410)]
Order 5485541814: assigned to [np.int64(5420)]
Order 5485628870: assigned to [np.int64(5490)]
Order 8029881327: assigned to [np.int64(5385)]
Order 8029889814: assigned to [np.int64(5420)]
Order 5484791041: assigned to [np.int64(5420)]
Order 5485298385: assigned to [np.int64(5490)]
Order 5485342422: assigned to [np.int64(5083)]
Order 5485490374: assigned to [np.int64(5490)]
Order 5485516497: assigned to [np.int64(5490)]
Order 5485531972: assigned to [np.int64(5410)]
Order 8029871781: assigned to [n

In [23]:
total_revenue = sum(price.get((o,s),0) * pulp.value(f[(o,s,d)]) for (o,s,d) in f)
total_penalty_paid = sum(penalty_rate.get((o,s),0) * price.get((o,s),0) * pulp.value(unfilled_by_line[(o,s)]) for (o,s) in unfilled_by_line)
total_shipping_paid = sum(ship_cost.get((o,d),0) * pulp.value(x[(o,d)]) for (o,d) in x)

print(f"Total revenue: {total_revenue:,.2f}")
print(f"Total penalty: {total_penalty_paid:,.2f}")
print(f"Total shipping: {total_shipping_paid:,.2f}")
print(f"Check: revenue - penalty - shipping = {total_revenue - total_penalty_paid - total_shipping_paid:,.2f}")
print(f"Reported objective value: {pulp.value(prob.objective):,.2f}")


Total revenue: 2,225,355.60
Total penalty: 873.94
Total shipping: 35,582.00
Check: revenue - penalty - shipping = 2,188,899.66
Reported objective value: 2,188,899.66


In [24]:
prob.solve()
print("Status:", pulp.LpStatus[prob.status])
print("Objective value (optimized profit):", pulp.value(prob.objective))

Status: Optimal
Objective value (optimized profit): 2188899.6639999994


In [25]:
N_ORDERS = 300
sample_orders = df_order['Group_Flag'].unique()[:N_ORDERS]

df_sample = df_baseline_ship[df_baseline_ship['Group_Flag'].isin(sample_orders)].copy()
df_sample_clean = df_sample.copy()
df_sample_clean['Order_SKU_Revenue'] = df_sample_clean['Order_SKU_Revenue'].fillna(0)
df_sample_clean['Penaltyforpotentialcuts'] = df_sample_clean['Penaltyforpotentialcuts'].fillna(0)
df_sample_clean['OrderedQty_converted'] = df_sample_clean['OrderedQty_converted'].fillna(0)
df_sample_clean['price_per_case'] = df_sample_clean['Order_SKU_Revenue'] / df_sample_clean['OrderedQty_converted']
df_sample_clean['price_per_case'] = df_sample_clean['price_per_case'].fillna(0)

sample_skus = df_sample_clean['MaterialNumber'].unique()
sample_dates = df_sample_clean['transportationplanningdate'].unique()

candidate_dcs = df_capacity_renamed[
    (df_capacity_renamed['MaterialNumber'].isin(sample_skus)) &
    (df_capacity_renamed['transportationplanningdate'].isin(sample_dates))
][['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
candidate_dcs['Available_inventory'] = candidate_dcs['Available_inventory'].clip(lower=0)

order_zips = df_sample_clean[['Group_Flag','ZipCode']].drop_duplicates()
ship_lookup = df_ship[df_ship['Plant'].isin(candidate_dcs['Plant'].unique())][['Plant','TargetZip','Shipping_Cost']]
sample_ship_costs = order_zips.merge(ship_lookup, left_on='ZipCode', right_on='TargetZip', how='left')
sample_ship_costs['Shipping_Cost'] = sample_ship_costs['Shipping_Cost'].fillna(0)

orders = df_sample_clean['Group_Flag'].unique()
dcs = candidate_dcs['Plant'].unique()

print(f"N_ORDERS = {N_ORDERS}")
print("Order-SKU lines:", df_sample_clean.shape[0])
print("Candidate DCs:", len(dcs))

N_ORDERS = 300
Order-SKU lines: 6801
Candidate DCs: 12


In [26]:
import pulp
from collections import defaultdict

prob = pulp.LpProblem("DOM_Sample_100", pulp.LpMaximize)

x = {}
for o in orders:
    for d in dcs:
        x[(o,d)] = pulp.LpVariable(f"x_{o}_{d}", cat='Binary')

f = {}
for _, row in df_sample_clean.iterrows():
    o = row['Group_Flag']
    s = row['MaterialNumber']
    for d in dcs:
        f[(o,s,d)] = pulp.LpVariable(f"f_{o}_{s}_{d}", lowBound=0, cat='Continuous')

print("x variables:", len(x))
print("f variables:", len(f))

x variables: 3600
f variables: 81612


In [27]:
price = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['price_per_case'].to_dict()
penalty_rate = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['Penaltyforpotentialcuts'].to_dict()
demand = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['OrderedQty_converted'].to_dict()
ship_cost = sample_ship_costs.set_index(['Group_Flag','Plant'])['Shipping_Cost'].to_dict()

revenue = pulp.lpSum(
    price.get((o,s), 0) * f[(o,s,d)] for (o,s,d) in f
)

unfilled_by_line = {}
for (o,s) in set((k[0],k[1]) for k in f):
    unfilled_by_line[(o,s)] = demand.get((o,s),0) - pulp.lpSum(f[(o,s,d)] for d in dcs if (o,s,d) in f)

penalty = pulp.lpSum(
    penalty_rate.get((o,s),0) * price.get((o,s),0) * unfilled_by_line[(o,s)]
    for (o,s) in unfilled_by_line
)

shipping = pulp.lpSum(
    ship_cost.get((o,d), 0) * x[(o,d)]
    for (o,d) in x
)

prob += revenue - penalty - shipping
print("Objective function set.")

Objective function set.


In [28]:
# Constraint 1: one DC per order
for o in orders:
    prob += pulp.lpSum(x[(o,d)] for d in dcs) == 1, f"OneDC_{o}"

# Constraint 2: fulfillment only at assigned DC
for (o,s,d) in f:
    prob += f[(o,s,d)] <= demand.get((o,s),0) * x[(o,d)], f"OnlyIfAssigned_{o}_{s}_{d}"

# Constraint 3: cannot exceed ordered quantity
for (o,s) in unfilled_by_line:
    prob += pulp.lpSum(f[(o,s,d)] for d in dcs if (o,s,d) in f) <= demand.get((o,s),0), f"MaxDemand_{o}_{s}"

# Constraint 4: inventory (Available_inventory already clipped >= 0 in candidate_dcs)
inv_lookup = candidate_dcs.set_index(['Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()
inv_groups = defaultdict(list)
for (o,s,d) in f:
    date = df_sample_clean[(df_sample_clean['Group_Flag']==o) & (df_sample_clean['MaterialNumber']==s)]['transportationplanningdate'].iloc[0]
    inv_groups[(s,d,date)].append((o,s,d))
for (s,d,date), var_list in inv_groups.items():
    available = inv_lookup.get((d,s,date), 0)
    prob += pulp.lpSum(f[k] for k in var_list) <= available, f"Inventory_{s}_{d}_{date}"

# Constraint 5: dock capacity (only where capacity > 0)
dock_lookup = df_dock.set_index(['Plant','Date'])['Dock_Capacity'].to_dict()
order_dates = df_sample_clean.set_index('Group_Flag')['transportationplanningdate'].to_dict()
dock_groups = defaultdict(list)
for (o,d) in x:
    date = order_dates[o]
    dock_groups[(d,date)].append((o,d))
for (d,date), var_list in dock_groups.items():
    capacity = dock_lookup.get((d,date), None)
    if capacity is not None and capacity > 0:
        prob += pulp.lpSum(x[k] for k in var_list) <= capacity, f"Dock_{d}_{date}"

print("All constraints added.")

All constraints added.


In [29]:
import time
start = time.time()
prob.solve()
elapsed = time.time() - start

print("Status:", pulp.LpStatus[prob.status])
print("Objective value:", pulp.value(prob.objective))
print(f"Solve time: {elapsed:.1f} seconds")

Status: Optimal
Objective value: 23198220.237666313
Solve time: 44.3 seconds


In [30]:
total_filled = sum(pulp.value(f[k]) for k in f if pulp.value(f[k]) > 0)
total_demand = df_sample_clean['OrderedQty_converted'].sum()
fill_rate = total_filled / total_demand

# confirm every order got exactly one DC
orders_with_one_dc = sum(1 for o in orders if sum(pulp.value(x[(o,d)]) for d in dcs) == 1)

print(f"Total filled: {total_filled:,.0f} / {total_demand:,.0f} ({fill_rate:.2%})")
print(f"Orders with exactly one DC assigned: {orders_with_one_dc} / {len(orders)}")

Total filled: 633,696 / 653,353 (96.99%)
Orders with exactly one DC assigned: 300 / 300


### The QUBO assisted Formulation

In [31]:

N_ORDERS = 20
sample_orders = df_order['Group_Flag'].unique()[:N_ORDERS]

df_sample = df_baseline_ship[df_baseline_ship['Group_Flag'].isin(sample_orders)].copy()
df_sample_clean = df_sample.copy()
df_sample_clean['Order_SKU_Revenue'] = df_sample_clean['Order_SKU_Revenue'].fillna(0)
df_sample_clean['Penaltyforpotentialcuts'] = df_sample_clean['Penaltyforpotentialcuts'].fillna(0)
df_sample_clean['OrderedQty_converted'] = df_sample_clean['OrderedQty_converted'].fillna(0)
df_sample_clean['price_per_case'] = (df_sample_clean['Order_SKU_Revenue'] / df_sample_clean['OrderedQty_converted']).fillna(0)

sample_skus = df_sample_clean['MaterialNumber'].unique()
sample_dates = df_sample_clean['transportationplanningdate'].unique()

candidate_dcs = df_capacity_renamed[
    (df_capacity_renamed['MaterialNumber'].isin(sample_skus)) &
    (df_capacity_renamed['transportationplanningdate'].isin(sample_dates))
][['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
candidate_dcs['Available_inventory'] = candidate_dcs['Available_inventory'].clip(lower=0)

order_zips = df_sample_clean[['Group_Flag','ZipCode']].drop_duplicates()
ship_lookup = df_ship[df_ship['Plant'].isin(candidate_dcs['Plant'].unique())][['Plant','TargetZip','Shipping_Cost']]
sample_ship_costs = order_zips.merge(ship_lookup, left_on='ZipCode', right_on='TargetZip', how='left')
sample_ship_costs['Shipping_Cost'] = sample_ship_costs['Shipping_Cost'].fillna(0)

orders = df_sample_clean['Group_Flag'].unique()
dcs = candidate_dcs['Plant'].unique()

print(f"N_ORDERS = {N_ORDERS} | lines: {df_sample_clean.shape[0]} | DCs: {len(dcs)}")

N_ORDERS = 20 | lines: 553 | DCs: 12


In [32]:
# Step 2: Precompute c_{o,d} — profit estimate per order-DC pair
inv_lookup = candidate_dcs.set_index(['Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()
ship_cost = sample_ship_costs.set_index(['Group_Flag','Plant'])['Shipping_Cost'].to_dict()

c = {}  # c[(o,d)] = estimated profit if order o assigned entirely to DC d

for o in orders:
    order_lines = df_sample_clean[df_sample_clean['Group_Flag'] == o]
    for d in dcs:
        profit = 0
        for _, row in order_lines.iterrows():
            s = row['MaterialNumber']
            date = row['transportationplanningdate']
            demand_qty = row['OrderedQty_converted']
            price = row['price_per_case']
            penalty_rate = row['Penaltyforpotentialcuts']
            
            avail = inv_lookup.get((d, s, date), 0)
            filled = min(demand_qty, avail)
            unfilled = demand_qty - filled
            
            profit += price * filled - penalty_rate * price * unfilled
        
        profit -= ship_cost.get((o, d), 0)
        c[(o, d)] = profit

print("Number of (order, DC) profit estimates:", len(c))
print("Sample values:", list(c.items())[:5])

Number of (order, DC) profit estimates: 240
Sample values: [((np.int64(5484913123), np.int64(5083)), 103593.0), ((np.int64(5484913123), np.int64(5159)), -1939.0), ((np.int64(5484913123), np.int64(5385)), -1165.0), ((np.int64(5484913123), np.int64(5410)), -1284.0), ((np.int64(5484913123), np.int64(5420)), -1048.0)]


In [33]:
# Step 3: Choose a penalty weight λ
max_abs_profit = max(abs(v) for v in c.values())
lambda_penalty = max_abs_profit * 3  # comfortably dominates any single profit term

print("Max |profit|:", max_abs_profit)
print("Chosen lambda:", lambda_penalty)

Max |profit|: 343452.19999999995
Chosen lambda: 1030356.5999999999


In [34]:
#Step 4: Build the QUBO dictionary

Q = {}

# --- Profit terms (negated, since QUBO minimizes) ---
for (o,d), profit in c.items():
    key = (o,d)
    Q[(key,key)] = Q.get((key,key), 0) - profit

# --- Penalty terms: one DC per order ---
for o in orders:
    dcs_for_order = [(o,d) for d in dcs]
    # linear part: -lambda for each x_{o,d}
    for key in dcs_for_order:
        Q[(key,key)] = Q.get((key,key), 0) - lambda_penalty
    # quadratic part: +2*lambda for each pair (d, d') where d < d'
    for i in range(len(dcs_for_order)):
        for j in range(i+1, len(dcs_for_order)):
            key_pair = (dcs_for_order[i], dcs_for_order[j])
            Q[key_pair] = Q.get(key_pair, 0) + 2*lambda_penalty

print("Number of QUBO terms:", len(Q))

Number of QUBO terms: 1560


Simulated Annealing Testing

In [35]:
# Step 5: Run simulated annealing on the QUBO
import neal

sampler = neal.SimulatedAnnealingSampler()
response = sampler.sample_qubo(Q, num_reads=1000,num_sweeps=1000)

best_sample = response.first.sample
best_energy = response.first.energy

print("Best energy (should be minimized):", best_energy)
print("Number of variables set to 1:", sum(v for v in best_sample.values()))

Best energy (should be minimized): -22648165.78521371
Number of variables set to 1: 20


In [36]:
# Step 6: Extract assignments and compute results
qubo_assignments = {}
for (o,d), val in best_sample.items():
    if val == 1:
        qubo_assignments[o] = d

print("Orders assigned:", len(qubo_assignments))
for o, d in list(qubo_assignments.items())[:5]:
    print(f"Order {o} -> DC {d}")

Orders assigned: 20
Order 5484495964 -> DC 5620
Order 5484791041 -> DC 5385
Order 5484845895 -> DC 5420
Order 5484913123 -> DC 5773
Order 5484930889 -> DC 5420


In [37]:
# Step 9: Compute fill rate, penalty, and profit for the QUBO solution

total_qubo_profit = sum(c[(o, qubo_assignments[o])] for o in qubo_assignments)

# recompute fill details per order for fill-rate reporting
qubo_filled = 0
qubo_demand_total = 0

for o in qubo_assignments:
    d = qubo_assignments[o]
    order_lines = df_sample_clean[df_sample_clean['Group_Flag'] == o]
    for _, row in order_lines.iterrows():
        s = row['MaterialNumber']
        date = row['transportationplanningdate']
        demand_qty = row['OrderedQty_converted']
        avail = inv_lookup.get((d, s, date), 0)
        filled = min(demand_qty, avail)
        qubo_filled += filled
        qubo_demand_total += demand_qty

qubo_fill_rate = qubo_filled / qubo_demand_total

print(f"QUBO total profit: {total_qubo_profit:,.2f}")
print(f"QUBO fill rate: {qubo_filled:,.0f} / {qubo_demand_total:,.0f} ({qubo_fill_rate:.2%})")

QUBO total profit: 2,041,033.79
QUBO fill rate: 47,500 / 54,744 (86.77%)


In [38]:
# True optimum for THIS QUBO structure: best DC per order, independently
optimal_profit = 0
optimal_assignments = {}
for o in orders:
    best_d = max(dcs, key=lambda d: c[(o,d)])
    optimal_assignments[o] = best_d
    optimal_profit += c[(o, best_d)]

print(f"True QUBO-optimal profit (best DC per order, independently): {optimal_profit:,.2f}")
print(f"Simulated annealing found: {total_qubo_profit:,.2f}")
print(f"Gap: {optimal_profit - total_qubo_profit:,.2f}")

# how many orders differ from the SA result?
diffs = sum(1 for o in orders if qubo_assignments.get(o) != optimal_assignments[o])
print(f"Orders where SA picked a different DC than the true optimum: {diffs} / {len(orders)}")

True QUBO-optimal profit (best DC per order, independently): 2,188,899.66
Simulated annealing found: 2,041,033.79
Gap: 147,865.88
Orders where SA picked a different DC than the true optimum: 17 / 20


## Solver Tuning Notes

Before settling on the final approach, several simulated annealing
configurations were tested manually:
- `num_reads=100` (default): large gap from optimum (~289K)
- `num_reads=1000` with custom `beta_range=[0.1,10]`: performed worse
  (~466K gap) — an unsuitable cooling schedule for this QUBO's energy scale
- `num_reads=1000` alone: notably better, but results varied meaningfully
  run-to-run (gap ranged ~109K–182K across repeated identical runs) due to
  simulated annealing's inherent randomness
- `num_reads=1000, num_sweeps=1000`: no consistent improvement over
  `num_reads=1000` alone

Given this run-to-run variability, the final reported results use 5
independent runs per method (SA and Tabu), summarized by mean/best/worst,
rather than a single run — see below.

In [39]:
import numpy as np

sa_profits = []
for trial in range(5):
    response = sampler.sample_qubo(Q, num_reads=1000)
    sample = response.first.sample
    assignments = {o: d for (o,d), val in sample.items() if val == 1}
    profit = sum(c[(o, assignments[o])] for o in assignments)
    sa_profits.append(profit)

print("SA profits across 5 runs:", [f"{p:,.2f}" for p in sa_profits])
print(f"SA mean: {np.mean(sa_profits):,.2f}, best: {max(sa_profits):,.2f}, worst: {min(sa_profits):,.2f}")

SA profits across 5 runs: ['2,036,417.56', '2,061,160.39', '2,049,972.38', '2,015,669.74', '2,007,841.28']
SA mean: 2,034,212.27, best: 2,061,160.39, worst: 2,007,841.28


In [40]:
import tabu

tabu_sampler = tabu.TabuSampler()
tabu_profits = []
for trial in range(5):
    tabu_response = tabu_sampler.sample_qubo(Q, num_reads=100)
    tabu_sample = tabu_response.first.sample
    tabu_assignments_trial = {o: d for (o,d), val in tabu_sample.items() if val == 1}
    profit = sum(c[(o, tabu_assignments_trial[o])] for o in tabu_assignments_trial)
    tabu_profits.append(profit)

print("Tabu profits across 5 runs:", [f"{p:,.2f}" for p in tabu_profits])
print(f"Tabu mean: {np.mean(tabu_profits):,.2f}, best: {max(tabu_profits):,.2f}, worst: {min(tabu_profits):,.2f}")

Tabu profits across 5 runs: ['1,996,935.85', '2,057,136.66', '2,086,127.57', '2,109,178.38', '2,091,114.49']
Tabu mean: 2,068,098.59, best: 2,109,178.38, worst: 1,996,935.85


## Extension: Adding Inventory Coupling via CQM

The QUBO above treats each order's DC assignment independently — there is no
shared constraint preventing multiple orders from being assigned to the same
DC and collectively over-consuming that DC's inventory for a shared SKU/date.
This section extends the model to include real inventory coupling across
orders, closing that gap with the full MILP formulation (Task 3).

Rather than hand-deriving slack variables to encode the inventory inequality
constraint (`Σ_o demand · x_{o,d} ≤ available_inventory`) directly into a QUBO
— a manual process that is easy to get subtly wrong — this section uses
`dimod`'s `ConstrainedQuadraticModel` (CQM), which allows constraints to be
written in natural inequality form, and then converts the resulting CQM into
a BQM (`cqm_to_bqm`) that `neal`/`tabu` can solve exactly as before. This
avoids depending on D-Wave's cloud-hosted CQM solver, which is no longer
available on free-tier accounts, while still getting the benefit of
`dimod` handling the slack-variable encoding automatically and correctly.

In [41]:
# Build the CQM with real inventory coupling
from dimod import ConstrainedQuadraticModel, Binary, quicksum

cqm = ConstrainedQuadraticModel()

# Same binary variables as before: x[(o,d)]
x_cqm = {(o,d): Binary(f"x_{o}_{d}") for o in orders for d in dcs}

# Objective: maximize profit = minimize negative profit
cqm.set_objective(-quicksum(c[(o,d)] * x_cqm[(o,d)] for (o,d) in x_cqm))

# Constraint 1: exactly one DC per order
for o in orders:
    cqm.add_constraint(
        quicksum(x_cqm[(o,d)] for d in dcs) == 1,
        label=f"OneDC_{o}"
    )

print("Objective and one-DC constraints added.")
print("Number of constraints so far:", len(cqm.constraints))

Objective and one-DC constraints added.
Number of constraints so far: 20


In [42]:
# Build a lookup: for each order, what SKUs/quantities does it need, and on what date
order_sku_demand = df_sample_clean.groupby(['Group_Flag','MaterialNumber','transportationplanningdate'])['OrderedQty_converted'].sum().reset_index()

# Group by (SKU, date) to find which orders compete for the same resource
inventory_constraints_added = 0

for (s, date), group in order_sku_demand.groupby(['MaterialNumber','transportationplanningdate']):
    relevant_orders = group['Group_Flag'].tolist()
    demand_lookup = dict(zip(group['Group_Flag'], group['OrderedQty_converted']))
    
    for d in dcs:
        available = inv_lookup.get((d, s, date), 0)
        
        # Only add constraint if any relevant order actually has x_cqm[(o,d)] defined
        terms = [(o,d) for o in relevant_orders if (o,d) in x_cqm]
        if terms:
            cqm.add_constraint(
                quicksum(demand_lookup[o] * x_cqm[(o,d)] for (o,d) in terms) <= available,
                label=f"Inv_{s}_{d}_{date}"
            )
            inventory_constraints_added += 1

print("Inventory constraints added:", inventory_constraints_added)
print("Total constraints now:", len(cqm.constraints))

Inventory constraints added: 6420
Total constraints now: 6440


In [43]:
# Convert the CQM to BQM  
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='dimod')

from dimod import cqm_to_bqm
bqm, invert = cqm_to_bqm(cqm, lagrange_multiplier=10*max_abs_profit)

print("Number of BQM variables (includes slack variables):", len(bqm.variables))

Number of BQM variables (includes slack variables): 617


In [44]:
# How many inventory constraints were actually "tight" enough to matter?
kept_constraints = len(cqm.constraints)
dropped_in_bqm = kept_constraints - (617 - 20 - 1200)  # rough estimate, adjust as needed
print(f"Total CQM constraints: {kept_constraints}")
print(f"Final BQM variables: {len(bqm.variables)}")

Total CQM constraints: 6440
Final BQM variables: 617


In [45]:
import neal
bqm, invert = cqm_to_bqm(cqm, lagrange_multiplier=100 * max_abs_profit)
response = sampler.sample(bqm, num_reads=5000)

# convert back to original CQM variable space
best_sample = response.first.sample
cqm_sample = invert(best_sample)

# extract assignments
coupled_assignments = {}
for var_name, val in cqm_sample.items():
    if val == 1 and var_name.startswith('x_'):
        parts = var_name.split('_')
        o = int(parts[1])
        d = int(parts[2])
        coupled_assignments[o] = d
print("Orders assigned:", len(coupled_assignments))

Orders assigned: 11


In [46]:
# Check how many CQM constraints are violated by this solution
violations = cqm.iter_violations(cqm_sample, skip_satisfied=True)
violation_list = list(violations)
print("Number of violated constraints:", len(violation_list))
for label, violation in violation_list[:10]:
    print(label, violation)

Number of violated constraints: 9
OneDC_5484913123 1.0
OneDC_5484930889 1.0
OneDC_5484845895 1.0
OneDC_5485541814 1.0
OneDC_8029889814 1.0
OneDC_5485298385 1.0
OneDC_5485342422 1.0
OneDC_5485531972 1.0
OneDC_8029871781 1.0


In [47]:
import tabu

tabu_sampler = tabu.TabuSampler()
tabu_response = tabu_sampler.sample(bqm, num_reads=200)
tabu_best = tabu_response.first.sample
tabu_cqm_sample = invert(tabu_best)

tabu_assignments = {}
for var_name, val in tabu_cqm_sample.items():
    if val == 1 and var_name.startswith('x_'):
        parts = var_name.split('_')
        tabu_assignments[int(parts[1])] = int(parts[2])

print("Tabu orders assigned:", len(tabu_assignments))
tabu_violations = list(cqm.iter_violations(tabu_cqm_sample, skip_satisfied=True))
print("Tabu violations:", len(tabu_violations))

Tabu orders assigned: 11
Tabu violations: 9


In [48]:
for o in [5484913123, 5484930889, 5484845895, 5485541814, 8029889814, 5485298385, 5485342422, 5485531972, 8029871781]:
    best_profit_available = max(c[(o,d)] for d in dcs)
    print(f"Order {o}: best possible profit across all DCs = {best_profit_available:,.2f}")

print(f"\nCurrent penalty (lagrange_multiplier): {100*max_abs_profit:,.2f}")

Order 5484913123: best possible profit across all DCs = 112,302.00
Order 5484930889: best possible profit across all DCs = 114,151.17
Order 5484845895: best possible profit across all DCs = 229,532.00
Order 5485541814: best possible profit across all DCs = 196,308.30
Order 8029889814: best possible profit across all DCs = 343,452.20
Order 5485298385: best possible profit across all DCs = 136,040.00
Order 5485342422: best possible profit across all DCs = 218,660.00
Order 5485531972: best possible profit across all DCs = 16,426.24
Order 8029871781: best possible profit across all DCs = 130,516.00

Current penalty (lagrange_multiplier): 34,345,220.00


In [49]:
worst_case_penalty = max(
    row['Penaltyforpotentialcuts'] * row['price_per_case'] * row['OrderedQty_converted']
    for _, row in df_sample_clean.iterrows()
)
print(f"Worst-case single-line penalty: {worst_case_penalty:,.2f}")

new_lagrange = 10 * worst_case_penalty  # scale to the real worst case, not just profit
bqm, invert = cqm_to_bqm(cqm, lagrange_multiplier=new_lagrange)

Worst-case single-line penalty: 1,211.36


In [50]:
o_test = 5484913123
for d in dcs:
    var_name = f"x_{o_test}_{d}"
    val = cqm_sample.get(var_name, "MISSING")
    print(f"{var_name}: {val}")

x_5484913123_5083: 0
x_5484913123_5159: 0
x_5484913123_5385: 0
x_5484913123_5410: 0
x_5484913123_5420: 0
x_5484913123_5490: 0
x_5484913123_5620: 0
x_5484913123_5641: 0
x_5484913123_5773: 0
x_5484913123_5081: 0
x_5484913123_5423: 0
x_5484913123_5376: 0


In [51]:
for d in dcs:
    var_name = f"x_5484913123_{d}"
    if var_name in bqm.variables:
        linear_bias = bqm.get_linear(var_name)
        print(f"{var_name}: linear bias = {linear_bias:,.2f}")
    else:
        print(f"{var_name}: NOT IN BQM")

x_5484913123_5083: linear bias = 186,797,141.40
x_5484913123_5159: linear bias = 6,759,487,647.80
x_5484913123_5385: linear bias = 6,759,486,873.80
x_5484913123_5410: linear bias = 6,759,486,992.80
x_5484913123_5420: linear bias = 6,759,486,756.80
x_5484913123_5490: linear bias = 6,759,487,800.80
x_5484913123_5620: linear bias = 6,759,490,267.80
x_5484913123_5641: linear bias = 6,759,486,853.80
x_5484913123_5773: linear bias = 12,352,592.40
x_5484913123_5081: linear bias = 6,759,490,267.80
x_5484913123_5423: linear bias = 6,759,490,655.80
x_5484913123_5376: linear bias = 6,759,486,908.80


In [52]:
print("Is x_5484913123_5083 in cqm objective?", 'x_5484913123_5083' in [v for v in cqm.objective.variables])
print("Objective coefficient:", cqm.objective.get_linear('x_5484913123_5083') if 'x_5484913123_5083' in cqm.objective.variables else "MISSING")

Is x_5484913123_5083 in cqm objective? True
Objective coefficient: -103593.0


### Aggregate SKUs into one inventory constraint per (order, DC)

In [53]:
from dimod import ConstrainedQuadraticModel, Binary, quicksum

cqm_v2 = ConstrainedQuadraticModel()
x_cqm_v2 = {(o,d): Binary(f"x_{o}_{d}") for o in orders for d in dcs}

cqm_v2.set_objective(-quicksum(c[(o,d)] * x_cqm_v2[(o,d)] for (o,d) in x_cqm_v2))

# Constraint 1: one DC per order (unchanged)
for o in orders:
    cqm_v2.add_constraint(
        quicksum(x_cqm_v2[(o,d)] for d in dcs) == 1,
        label=f"OneDC_{o}"
    )

# Constraint 2 (AGGREGATED): total demand across all SKUs vs total available inventory, per (order, DC)
inventory_constraints_v2 = 0
for o in orders:
    order_lines = df_sample_clean[df_sample_clean['Group_Flag'] == o]
    date = order_dates[o]
    total_demand = order_lines['OrderedQty_converted'].sum()
    
    for d in dcs:
        total_available = sum(
            inv_lookup.get((d, row['MaterialNumber'], date), 0)
            for _, row in order_lines.iterrows()
        )
        # constraint: if assigned to this DC, demand can't exceed available inventory
        cqm_v2.add_constraint(
            total_demand * x_cqm_v2[(o,d)] <= total_available,
            label=f"Inv_{o}_{d}"
        )
        inventory_constraints_v2 += 1

print("Aggregated inventory constraints:", inventory_constraints_v2)
print("Total constraints:", len(cqm_v2.constraints))

Aggregated inventory constraints: 240
Total constraints: 260


In [54]:
bqm_v2, invert_v2 = cqm_to_bqm(cqm_v2, lagrange_multiplier=100 * max_abs_profit)

print("Number of BQM v2 variables:", len(bqm_v2.variables))

Number of BQM v2 variables: 367


In [55]:
response_v2 = sampler.sample(bqm_v2, num_reads=1000)
best_sample_v2 = response_v2.first.sample
cqm_sample_v2 = invert_v2(best_sample_v2)

coupled_assignments_v2 = {}
for var_name, val in cqm_sample_v2.items():
    if val == 1 and var_name.startswith('x_'):
        parts = var_name.split('_')
        o = int(parts[1])
        d = int(parts[2])
        coupled_assignments_v2[o] = d

print("Orders assigned:", len(coupled_assignments_v2))

violations_v2 = list(cqm_v2.iter_violations(cqm_sample_v2, skip_satisfied=True))
print("Number of violated constraints:", len(violations_v2))

Orders assigned: 20
Number of violated constraints: 0


In [56]:
total_coupled_profit = sum(c[(o, coupled_assignments_v2[o])] for o in coupled_assignments_v2)
print(f"Coupled QUBO profit: {total_coupled_profit:,.2f}")

# recompute fill rate
coupled_filled = 0
coupled_demand_total = 0

for o in coupled_assignments_v2:
    d = coupled_assignments_v2[o]
    order_lines = df_sample_clean[df_sample_clean['Group_Flag'] == o]
    for _, row in order_lines.iterrows():
        s = row['MaterialNumber']
        date = row['transportationplanningdate']
        demand_qty = row['OrderedQty_converted']
        avail = inv_lookup.get((d, s, date), 0)
        filled = min(demand_qty, avail)
        coupled_filled += filled
        coupled_demand_total += demand_qty

coupled_fill_rate = coupled_filled / coupled_demand_total
print(f"Coupled QUBO fill rate: {coupled_filled:,.0f} / {coupled_demand_total:,.0f} ({coupled_fill_rate:.2%})")

Coupled QUBO profit: 2,142,898.60
Coupled QUBO fill rate: 52,002 / 54,744 (94.99%)


## Testing the Formulation For Full Data Set provided by Nestle

In [57]:
N_ORDERS = len(df_order['Group_Flag'].unique())  # all 1,109 orders
print("Running on full dataset:", N_ORDERS, "orders")

sample_orders = df_order['Group_Flag'].unique()[:N_ORDERS]

df_sample = df_baseline_ship[df_baseline_ship['Group_Flag'].isin(sample_orders)].copy()
df_sample_clean = df_sample.copy()
df_sample_clean['Order_SKU_Revenue'] = df_sample_clean['Order_SKU_Revenue'].fillna(0)
df_sample_clean['Penaltyforpotentialcuts'] = df_sample_clean['Penaltyforpotentialcuts'].fillna(0)
df_sample_clean['OrderedQty_converted'] = df_sample_clean['OrderedQty_converted'].fillna(0)
df_sample_clean['price_per_case'] = (df_sample_clean['Order_SKU_Revenue'] / df_sample_clean['OrderedQty_converted']).fillna(0)

sample_skus = df_sample_clean['MaterialNumber'].unique()
sample_dates = df_sample_clean['transportationplanningdate'].unique()

candidate_dcs = df_capacity_renamed[
    (df_capacity_renamed['MaterialNumber'].isin(sample_skus)) &
    (df_capacity_renamed['transportationplanningdate'].isin(sample_dates))
][['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
candidate_dcs['Available_inventory'] = candidate_dcs['Available_inventory'].clip(lower=0)

order_zips = df_sample_clean[['Group_Flag','ZipCode']].drop_duplicates()
ship_lookup = df_ship[df_ship['Plant'].isin(candidate_dcs['Plant'].unique())][['Plant','TargetZip','Shipping_Cost']]
sample_ship_costs = order_zips.merge(ship_lookup, left_on='ZipCode', right_on='TargetZip', how='left')
sample_ship_costs['Shipping_Cost'] = sample_ship_costs['Shipping_Cost'].fillna(0)

orders = df_sample_clean['Group_Flag'].unique()
dcs = candidate_dcs['Plant'].unique()

print(f"Order-SKU lines: {df_sample_clean.shape[0]}")
print(f"Candidate DCs: {len(dcs)}")

Running on full dataset: 1109 orders
Order-SKU lines: 25193
Candidate DCs: 12


In [58]:
print("Unique dates in full sample:", len(sample_dates))
print("Unique dates in full capacity data:", df_capacity_renamed['transportationplanningdate'].nunique())
print("Candidate_dcs rows:", candidate_dcs.shape[0])

Unique dates in full sample: 10
Unique dates in full capacity data: 32
Candidate_dcs rows: 61250


In [59]:
import time

prob_full = pulp.LpProblem("DOM_Full", pulp.LpMaximize)

x_full = {}
for o in orders:
    for d in dcs:
        x_full[(o,d)] = pulp.LpVariable(f"x_{o}_{d}", cat='Binary')

f_full = {}
for _, row in df_sample_clean.iterrows():
    o = row['Group_Flag']
    s = row['MaterialNumber']
    for d in dcs:
        f_full[(o,s,d)] = pulp.LpVariable(f"f_{o}_{s}_{d}", lowBound=0, cat='Continuous')

print("x variables:", len(x_full))
print("f variables:", len(f_full))

x variables: 13308
f variables: 302316


In [60]:
print("N_ORDERS:", N_ORDERS)
print("Number of unique sample_orders:", len(sample_orders))
print("Number of unique orders (final):", len(orders))

N_ORDERS: 1109
Number of unique sample_orders: 1109
Number of unique orders (final): 1109


In [61]:

price = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['price_per_case'].to_dict()
penalty_rate = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['Penaltyforpotentialcuts'].to_dict()
demand = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['OrderedQty_converted'].to_dict()
ship_cost = sample_ship_costs.set_index(['Group_Flag','Plant'])['Shipping_Cost'].to_dict()
date_lookup = df_sample_clean.set_index(['Group_Flag','MaterialNumber'])['transportationplanningdate'].to_dict()  # NEW — replaces the slow .iloc[0] filter

revenue = pulp.lpSum(price.get((o,s), 0) * f_full[(o,s,d)] for (o,s,d) in f_full)

unfilled_by_line = {}
for (o,s) in set((k[0],k[1]) for k in f_full):
    unfilled_by_line[(o,s)] = demand.get((o,s),0) - pulp.lpSum(f_full[(o,s,d)] for d in dcs if (o,s,d) in f_full)

penalty = pulp.lpSum(penalty_rate.get((o,s),0) * price.get((o,s),0) * unfilled_by_line[(o,s)] for (o,s) in unfilled_by_line)
shipping = pulp.lpSum(ship_cost.get((o,d), 0) * x_full[(o,d)] for (o,d) in x_full)

prob_full += revenue - penalty - shipping
print("Objective function set.") 

Objective function set.


In [62]:
# Constraint 1: one DC per order
for o in orders:
    prob_full += pulp.lpSum(x_full[(o,d)] for d in dcs) == 1, f"OneDC_{o}"

# Constraint 2: fulfillment only at assigned DC
for (o,s,d) in f_full:
    prob_full += f_full[(o,s,d)] <= demand.get((o,s),0) * x_full[(o,d)], f"OnlyIfAssigned_{o}_{s}_{d}"

# Constraint 3: cannot exceed ordered quantity
for (o,s) in unfilled_by_line:
    prob_full += pulp.lpSum(f_full[(o,s,d)] for d in dcs if (o,s,d) in f_full) <= demand.get((o,s),0), f"MaxDemand_{o}_{s}"

# Constraint 4: inventory — vectorized date lookup (the fix)
inv_lookup = candidate_dcs.set_index(['Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()
inv_groups = defaultdict(list)
for (o,s,d) in f_full:
    date = date_lookup[(o,s)]
    inv_groups[(s,d,date)].append((o,s,d))
for (s,d,date), var_list in inv_groups.items():
    available = inv_lookup.get((d,s,date), 0)
    prob_full += pulp.lpSum(f_full[k] for k in var_list) <= available, f"Inventory_{s}_{d}_{date}"

# Constraint 5: dock capacity (only where capacity > 0)
dock_lookup = df_dock.set_index(['Plant','Date'])['Dock_Capacity'].to_dict()
order_dates = df_sample_clean.set_index('Group_Flag')['transportationplanningdate'].to_dict()
dock_groups = defaultdict(list)
for (o,d) in x_full:
    date = order_dates[o]
    dock_groups[(d,date)].append((o,d))
for (d,date), var_list in dock_groups.items():
    capacity = dock_lookup.get((d,date), None)
    if capacity is not None and capacity > 0:
        prob_full += pulp.lpSum(x_full[k] for k in var_list) <= capacity, f"Dock_{d}_{date}"

print("All constraints added.")

All constraints added.


In [63]:
solver = pulp.PULP_CBC_CMD(msg=1, timeLimit=900, gapRel=0.01)  # 15 min cap, stop early if within 1% of optimal
start = time.time()
prob_full.solve(solver)
elapsed = time.time() - start

print("Status:", pulp.LpStatus[prob_full.status])
print("Objective value:", pulp.value(prob_full.objective))
print(f"Solve time: {elapsed:.1f} seconds")

KeyboardInterrupt: 

In [ ]:
total_demand = df_sample_clean['OrderedQty_converted'].sum()
total_filled = sum(pulp.value(f_full[k]) for k in f_full if pulp.value(f_full[k]) is not None)
fill_rate = total_filled / total_demand

print(f"Total demand: {total_demand:,.0f}")
print(f"Total filled: {total_filled:,.0f}")
print(f"Fill rate: {fill_rate*100:.2f}%")

### Aggregated-constraint scaling check
*(structure only — variable/constraint counts, no solving; uses a coarse
per-DC-total inventory aggregation for structural purposes only)*

In [64]:
import dimod
from dimod import ConstrainedQuadraticModel, Binary, cqm_to_bqm

def build_aggregated_cqm_variable_count(n_orders):
    sample_orders_n = df_order['Group_Flag'].unique()[:n_orders]
    df_n = df_baseline_ship[df_baseline_ship['Group_Flag'].isin(sample_orders_n)].copy()
    df_n['OrderedQty_converted'] = df_n['OrderedQty_converted'].fillna(0)

    orders_n = df_n['Group_Flag'].unique()
    skus_n = df_n['MaterialNumber'].unique()
    dates_n = df_n['transportationplanningdate'].unique()

    candidate_dcs_n = df_capacity_renamed[
        (df_capacity_renamed['MaterialNumber'].isin(skus_n)) &
        (df_capacity_renamed['transportationplanningdate'].isin(dates_n))
    ][['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
    candidate_dcs_n['Available_inventory'] = candidate_dcs_n['Available_inventory'].clip(lower=0)
    dcs_n = candidate_dcs_n['Plant'].unique()

    # x_{o,d} binary assignment variables
    x_vars = {(o, d): Binary(f"x_{o}_{d}") for o in orders_n for d in dcs_n}

    cqm = ConstrainedQuadraticModel()

    # Constraint 1: exactly one DC per order
    for o in orders_n:
        cqm.add_constraint(sum(x_vars[(o, d)] for d in dcs_n) == 1, label=f"OneDC_{o}")

    # Constraint 4 (aggregated): total demand per (order, DC) <= total inventory across that order's SKUs at that DC
    order_demand = df_n.groupby('Group_Flag')['OrderedQty_converted'].sum().to_dict()
    inv_by_dc = candidate_dcs_n.groupby('Plant')['Available_inventory'].sum().to_dict()  # coarse aggregate proxy
    for o in orders_n:
        for d in dcs_n:
            cqm.add_constraint(
                order_demand.get(o, 0) * x_vars[(o, d)] <= inv_by_dc.get(d, 0),
                label=f"Inv_{o}_{d}"
            )

    bqm, invert = cqm_to_bqm(cqm)
    return len(x_vars), cqm.num_constraints(), bqm.num_variables, len(bqm.quadratic)

for n in [20, 50, 100]:
    x_count, constraint_count, bqm_vars, bqm_interactions = build_aggregated_cqm_variable_count(n)
    print(f"N_ORDERS={n}: x_vars={x_count}, constraints={constraint_count}, BQM_vars={bqm_vars}, BQM_interactions={bqm_interactions}")

N_ORDERS=20: x_vars=240, constraints=260, BQM_vars=380, BQM_interactions=1880
N_ORDERS=50: x_vars=600, constraints=650, BQM_vars=950, BQM_interactions=4700
N_ORDERS=100: x_vars=1200, constraints=1300, BQM_vars=1200, BQM_interactions=6600


### Solved Results at Increasing Scale

*(actual solve, using the same accurate per-order-SKU inventory logic
validated in Section 05)*

In [65]:
import time

def build_and_solve_aggregated_cqm(n_orders, num_reads=1000):
    sample_orders_n = df_order['Group_Flag'].unique()[:n_orders]
    df_n = df_baseline_ship[df_baseline_ship['Group_Flag'].isin(sample_orders_n)].copy()
    df_n['Order_SKU_Revenue'] = df_n['Order_SKU_Revenue'].fillna(0)
    df_n['Penaltyforpotentialcuts'] = df_n['Penaltyforpotentialcuts'].fillna(0)
    df_n['OrderedQty_converted'] = df_n['OrderedQty_converted'].fillna(0)
    df_n['price_per_case'] = (df_n['Order_SKU_Revenue'] / df_n['OrderedQty_converted']).fillna(0)

    orders_n = df_n['Group_Flag'].unique()
    skus_n = df_n['MaterialNumber'].unique()
    dates_n = df_n['transportationplanningdate'].unique()

    candidate_dcs_n = df_capacity_renamed[
        (df_capacity_renamed['MaterialNumber'].isin(skus_n)) &
        (df_capacity_renamed['transportationplanningdate'].isin(dates_n))
    ][['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
    candidate_dcs_n['Available_inventory'] = candidate_dcs_n['Available_inventory'].clip(lower=0)
    dcs_n = candidate_dcs_n['Plant'].unique()
    inv_lookup_n = candidate_dcs_n.set_index(['Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()

    order_zips_n = df_n[['Group_Flag','ZipCode']].drop_duplicates()
    ship_lookup_n = df_ship[df_ship['Plant'].isin(dcs_n)][['Plant','TargetZip','Shipping_Cost']]
    ship_costs_n = order_zips_n.merge(ship_lookup_n, left_on='ZipCode', right_on='TargetZip', how='left')
    ship_costs_n['Shipping_Cost'] = ship_costs_n['Shipping_Cost'].fillna(0)
    ship_lookup_dict = ship_costs_n.set_index(['Group_Flag','Plant'])['Shipping_Cost'].to_dict()

    # precompute profit c[(o,d)], same logic as your validated independent-order QUBO
    order_dates_n = df_n.groupby('Group_Flag')['transportationplanningdate'].first().to_dict()
    c_n = {}
    for o in orders_n:
        order_lines = df_n[df_n['Group_Flag'] == o]
        date = order_dates_n[o]
        for d in dcs_n:
            profit = 0
            for _, row in order_lines.iterrows():
                s = row['MaterialNumber']
                demand_qty = row['OrderedQty_converted']
                price = row['price_per_case']
                penalty_rate = row['Penaltyforpotentialcuts']
                avail = inv_lookup_n.get((d, s, date), 0)
                filled = min(demand_qty, avail)
                unfilled = demand_qty - filled
                profit += price * filled - penalty_rate * price * unfilled
            profit -= ship_lookup_dict.get((o, d), 0)
            c_n[(o,d)] = profit
    max_abs_profit_n = max(abs(v) for v in c_n.values())

    # accurate aggregated inventory: per-order-SKU-specific totals
    x_vars = {(o, d): Binary(f"x_{o}_{d}") for o in orders_n for d in dcs_n}
    cqm_n = ConstrainedQuadraticModel()
    cqm_n.set_objective(-quicksum(c_n[(o,d)] * x_vars[(o,d)] for (o,d) in x_vars))

    for o in orders_n:
        cqm_n.add_constraint(quicksum(x_vars[(o,d)] for d in dcs_n) == 1, label=f"OneDC_{o}")

    for o in orders_n:
        order_lines = df_n[df_n['Group_Flag'] == o]
        date = order_dates_n[o]
        for d in dcs_n:
            total_avail = sum(inv_lookup_n.get((d, row['MaterialNumber'], date), 0) for _, row in order_lines.iterrows())
            total_demand = order_lines['OrderedQty_converted'].sum()
            cqm_n.add_constraint(total_demand * x_vars[(o,d)] <= total_avail, label=f"Inv_{o}_{d}")

    bqm_n, invert_n = cqm_to_bqm(cqm_n, lagrange_multiplier=100 * max_abs_profit_n)

    start = time.time()
    response_n = sampler.sample(bqm_n, num_reads=num_reads)
    solve_time = time.time() - start

    best_sample_n = response_n.first.sample
    cqm_sample_n = invert_n(best_sample_n)

    assignments_n = {}
    for var_name, val in cqm_sample_n.items():
        if val == 1 and var_name.startswith('x_'):
            parts = var_name.split('_')
            assignments_n[int(parts[1])] = int(parts[2])

    violations_n = list(cqm_n.iter_violations(cqm_sample_n, skip_satisfied=True))

    total_profit = sum(c_n[(o, assignments_n[o])] for o in assignments_n)
    filled = 0
    demand_total = 0
    for o in assignments_n:
        d = assignments_n[o]
        order_lines = df_n[df_n['Group_Flag'] == o]
        date = order_dates_n[o]
        for _, row in order_lines.iterrows():
            s = row['MaterialNumber']
            demand_qty = row['OrderedQty_converted']
            avail = inv_lookup_n.get((d, s, date), 0)
            filled += min(demand_qty, avail)
            demand_total += demand_qty

    return {
        'n_orders': n_orders,
        'x_vars': len(x_vars),
        'cqm_constraints': len(cqm_n.constraints),
        'bqm_vars': len(bqm_n.variables),
        'bqm_interactions': len(bqm_n.quadratic),
        'solve_time': solve_time,
        'orders_assigned': len(assignments_n),
        'violations': len(violations_n),
        'profit': total_profit,
        'fill_rate': filled/demand_total if demand_total else 0
    }

r = build_and_solve_aggregated_cqm(1109)
print(r)

{'n_orders': 1109, 'x_vars': 13308, 'cqm_constraints': 14417, 'bqm_vars': 17664, 'bqm_interactions': 96488, 'solve_time': 811.4720821380615, 'orders_assigned': 1104, 'violations': 18, 'profit': 71256655.02471766, 'fill_rate': 0.840395230669299}


In [66]:
def build_and_solve_aggregated_cqm_for_orders(orders_n, df_n, num_reads=1000):
    orders_n = list(orders_n)
    
    skus_n = df_n['MaterialNumber'].unique()
    dates_n = df_n['transportationplanningdate'].unique()

    candidate_dcs_n = df_capacity_renamed[
        (df_capacity_renamed['MaterialNumber'].isin(skus_n)) &
        (df_capacity_renamed['transportationplanningdate'].isin(dates_n))
    ][['Plant','MaterialNumber','transportationplanningdate','Available_inventory']].copy()
    candidate_dcs_n['Available_inventory'] = candidate_dcs_n['Available_inventory'].clip(lower=0)
    dcs_n = candidate_dcs_n['Plant'].unique()
    inv_lookup_n = candidate_dcs_n.set_index(['Plant','MaterialNumber','transportationplanningdate'])['Available_inventory'].to_dict()

    order_zips_n = df_n[['Group_Flag','ZipCode']].drop_duplicates()
    ship_lookup_n = df_ship[df_ship['Plant'].isin(dcs_n)][['Plant','TargetZip','Shipping_Cost']]
    ship_costs_n = order_zips_n.merge(ship_lookup_n, left_on='ZipCode', right_on='TargetZip', how='left')
    ship_costs_n['Shipping_Cost'] = ship_costs_n['Shipping_Cost'].fillna(0)
    ship_lookup_dict = ship_costs_n.set_index(['Group_Flag','Plant'])['Shipping_Cost'].to_dict()

    order_dates_n = df_n.groupby('Group_Flag')['transportationplanningdate'].first().to_dict()
    c_n = {}
    for o in orders_n:
        order_lines = df_n[df_n['Group_Flag'] == o]
        date = order_dates_n[o]
        for d in dcs_n:
            profit = 0
            for _, row in order_lines.iterrows():
                s = row['MaterialNumber']
                demand_qty = row['OrderedQty_converted']
                price = row['price_per_case']
                penalty_rate = row['Penaltyforpotentialcuts']
                avail = inv_lookup_n.get((d, s, date), 0)
                filled = min(demand_qty, avail)
                unfilled = demand_qty - filled
                profit += price * filled - penalty_rate * price * unfilled
            profit -= ship_lookup_dict.get((o, d), 0)
            c_n[(o,d)] = profit
    max_abs_profit_n = max(abs(v) for v in c_n.values())

    x_vars = {(o, d): Binary(f"x_{o}_{d}") for o in orders_n for d in dcs_n}
    cqm_n = ConstrainedQuadraticModel()
    cqm_n.set_objective(-quicksum(c_n[(o,d)] * x_vars[(o,d)] for (o,d) in x_vars))

    for o in orders_n:
        cqm_n.add_constraint(quicksum(x_vars[(o,d)] for d in dcs_n) == 1, label=f"OneDC_{o}")

    for o in orders_n:
        order_lines = df_n[df_n['Group_Flag'] == o]
        date = order_dates_n[o]
        for d in dcs_n:
            total_avail = sum(inv_lookup_n.get((d, row['MaterialNumber'], date), 0) for _, row in order_lines.iterrows())
            total_demand = order_lines['OrderedQty_converted'].sum()
            cqm_n.add_constraint(total_demand * x_vars[(o,d)] <= total_avail, label=f"Inv_{o}_{d}")

    bqm_n, invert_n = cqm_to_bqm(cqm_n, lagrange_multiplier=100 * max_abs_profit_n)

    start = time.time()
    response_n = sampler.sample(bqm_n, num_reads=num_reads)
    solve_time = time.time() - start

    best_sample_n = response_n.first.sample
    cqm_sample_n = invert_n(best_sample_n)

    assignments_n = {}
    for var_name, val in cqm_sample_n.items():
        if val == 1 and var_name.startswith('x_'):
            parts = var_name.split('_')
            assignments_n[int(parts[1])] = int(parts[2])

    violations_n = list(cqm_n.iter_violations(cqm_sample_n, skip_satisfied=True))

    total_profit = sum(c_n[(o, assignments_n[o])] for o in assignments_n)
    filled = 0
    demand_total = 0
    for o in assignments_n:
        d = assignments_n[o]
        order_lines = df_n[df_n['Group_Flag'] == o]
        date = order_dates_n[o]
        for _, row in order_lines.iterrows():
            s = row['MaterialNumber']
            demand_qty = row['OrderedQty_converted']
            avail = inv_lookup_n.get((d, s, date), 0)
            filled += min(demand_qty, avail)
            demand_total += demand_qty

    return {
        'x_vars': len(x_vars),
        'cqm_constraints': len(cqm_n.constraints),
        'bqm_vars': len(bqm_n.variables),
        'solve_time': solve_time,
        'orders_assigned': len(assignments_n),
        'violations': len(violations_n),
        'profit': total_profit,
        'fill_rate': filled/demand_total if demand_total else 0
    }

In [67]:
df_full = df_baseline_ship.copy()
df_full['Order_SKU_Revenue'] = df_full['Order_SKU_Revenue'].fillna(0)
df_full['Penaltyforpotentialcuts'] = df_full['Penaltyforpotentialcuts'].fillna(0)
df_full['OrderedQty_converted'] = df_full['OrderedQty_converted'].fillna(0)
df_full['price_per_case'] = (df_full['Order_SKU_Revenue'] / df_full['OrderedQty_converted']).fillna(0)

In [68]:
def solve_aggregated_cqm_by_date(df_full, num_reads=1000):
    all_results = []
    dates = df_full['transportationplanningdate'].unique()
    
    total_start = time.time()
    for date in dates:
        df_date = df_full[df_full['transportationplanningdate'] == date].copy()
        orders_date = df_date['Group_Flag'].unique()
        n_orders_date = len(orders_date)
        
        if n_orders_date == 0:
            continue
        
        result = build_and_solve_aggregated_cqm_for_orders(orders_date, df_date, num_reads=num_reads)
        result['date'] = date
        result['n_orders'] = n_orders_date
        all_results.append(result)
        print(f"Date {date}: {n_orders_date} orders, {result['violations']} violations, "
              f"{result['orders_assigned']}/{n_orders_date} assigned, {result['solve_time']:.1f}s")
    
    total_time = time.time() - total_start
    
    total_profit = sum(r['profit'] for r in all_results)
    total_assigned = sum(r['orders_assigned'] for r in all_results)
    total_violations = sum(r['violations'] for r in all_results)
    total_orders = sum(r['n_orders'] for r in all_results)
    
    print(f"\n=== COMBINED ACROSS ALL DATES ===")
    print(f"Total solve time: {total_time:.1f}s")
    print(f"Orders assigned: {total_assigned}/{total_orders}")
    print(f"Total violations: {total_violations}")
    print(f"Total profit: {total_profit:,.2f}")
    
    return all_results

results = solve_aggregated_cqm_by_date(df_full, num_reads=1000)
print(results)

Date 2024-06-26 00:00:00: 312 orders, 3 violations, 310/312 assigned, 153.7s
Date 2024-06-25 00:00:00: 186 orders, 0 violations, 186/186 assigned, 84.7s
Date 2024-06-24 00:00:00: 164 orders, 0 violations, 164/164 assigned, 75.7s
Date 2024-06-27 00:00:00: 235 orders, 2 violations, 233/235 assigned, 107.5s
Date 2024-07-02 00:00:00: 16 orders, 0 violations, 16/16 assigned, 6.8s
Date 2024-06-28 00:00:00: 112 orders, 1 violations, 111/112 assigned, 45.1s
Date 2024-07-01 00:00:00: 18 orders, 0 violations, 18/18 assigned, 7.0s
Date 2024-06-29 00:00:00: 20 orders, 0 violations, 20/20 assigned, 10.1s
Date 2024-06-30 00:00:00: 39 orders, 0 violations, 39/39 assigned, 14.4s
Date 2024-07-03 00:00:00: 7 orders, 0 violations, 7/7 assigned, 2.9s

=== COMBINED ACROSS ALL DATES ===
Total solve time: 572.4s
Orders assigned: 1104/1109
Total violations: 6
Total profit: 73,546,533.67
[{'x_vars': 3744, 'cqm_constraints': 4056, 'bqm_vars': 5075, 'solve_time': 153.7456340789795, 'orders_assigned': 310, 'viola

In [69]:
def solve_aggregated_cqm_by_date_v2(df_full, num_reads_default=1000, num_reads_large=3000, large_threshold=150):
    all_results = []
    dates = df_full['transportationplanningdate'].unique()
    
    total_start = time.time()
    for date in dates:
        df_date = df_full[df_full['transportationplanningdate'] == date].copy()
        orders_date = df_date['Group_Flag'].unique()
        n_orders_date = len(orders_date)
        
        if n_orders_date == 0:
            continue
        
        reads = num_reads_large if n_orders_date >= large_threshold else num_reads_default
        result = build_and_solve_aggregated_cqm_for_orders(orders_date, df_date, num_reads=reads)
        result['date'] = date
        result['n_orders'] = n_orders_date
        result['num_reads_used'] = reads
        all_results.append(result)
        print(f"Date {date}: {n_orders_date} orders, reads={reads}, {result['violations']} violations, "
              f"{result['orders_assigned']}/{n_orders_date} assigned, {result['solve_time']:.1f}s")
    
    total_time = time.time() - total_start
    total_profit = sum(r['profit'] for r in all_results)
    total_assigned = sum(r['orders_assigned'] for r in all_results)
    total_violations = sum(r['violations'] for r in all_results)
    total_orders = sum(r['n_orders'] for r in all_results)
    
    print(f"\n=== COMBINED ACROSS ALL DATES ===")
    print(f"Total solve time: {total_time:.1f}s")
    print(f"Orders assigned: {total_assigned}/{total_orders}")
    print(f"Total violations: {total_violations}")
    print(f"Total profit: {total_profit:,.2f}")
    
    return all_results

results_v2 = solve_aggregated_cqm_by_date_v2(df_full)

Date 2024-06-26 00:00:00: 312 orders, reads=3000, 2 violations, 310/312 assigned, 447.5s
Date 2024-06-25 00:00:00: 186 orders, reads=3000, 0 violations, 186/186 assigned, 253.0s
Date 2024-06-24 00:00:00: 164 orders, reads=3000, 0 violations, 164/164 assigned, 232.0s
Date 2024-06-27 00:00:00: 235 orders, reads=3000, 2 violations, 233/235 assigned, 316.5s
Date 2024-07-02 00:00:00: 16 orders, reads=1000, 0 violations, 16/16 assigned, 6.8s
Date 2024-06-28 00:00:00: 112 orders, reads=1000, 1 violations, 111/112 assigned, 45.3s
Date 2024-07-01 00:00:00: 18 orders, reads=1000, 0 violations, 18/18 assigned, 7.0s
Date 2024-06-29 00:00:00: 20 orders, reads=1000, 0 violations, 20/20 assigned, 9.9s
Date 2024-06-30 00:00:00: 39 orders, reads=1000, 0 violations, 39/39 assigned, 14.2s
Date 2024-07-03 00:00:00: 7 orders, reads=1000, 0 violations, 7/7 assigned, 2.8s

=== COMBINED ACROSS ALL DATES ===
Total solve time: 1401.4s
Orders assigned: 1104/1109
Total violations: 5
Total profit: 74,194,069.46
